In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, geopandas as gpd
from pathlib import Path

L = Path('/content/drive/MyDrive/Logika Ui/KTI_KARHUTLA/lainnya')
for p in sorted(L.iterdir()):
    print(p.name, round(p.stat().st_size / 1e6, 1), 'MB')

for p in sorted(L.glob('*.xlsx')):
    xl = pd.ExcelFile(p)
    print('\n=====', p.name, '| sheet:', xl.sheet_names)
    for s in xl.sheet_names[:3]:
        d = xl.parse(s, header=None, nrows=8)
        print('--', s, d.shape)
        print(d.to_string())

for p in sorted(L.glob('*.geojson')) + sorted(L.glob('*.json')):
    g = gpd.read_file(p, rows=5)
    print('\n=====', p.name)
    print('CRS:', g.crs)
    print(list(g.columns))
    print(g.drop(columns='geometry').to_string())
    print(g.geom_type.unique())

Mounted at /content/drive
Indonesia_Peat_Lands.geojson 31.9 MB
luas-karhutla-kota.xlsx 0.3 MB
luas-karhutla-provinsi.xlsx 0.0 MB
sebaran-hotspot.xlsx 1.2 MB

===== luas-karhutla-kota.xlsx | sheet: ['data']
-- data (8, 14)
             0                1       2         3         4       5       6      7       8       9       10        11        12      13
0  Kode Wilayah        Kabupaten    2015      2016      2017    2018    2019   2020    2021    2022    2023      2024      2025    2026
1          1105       ACEH BARAT       0    537,00  1.393,00   13,00   86,00      0   41,00  134,00  321,30    141,92  1.091,18  844,87
2          1112  ACEH BARAT DAYA   98,00    179,00         0       0       0  47,00       0  173,00   10,64    156,64    232,47   95,67
3          1106       ACEH BESAR       0    280,00    816,00       0  161,00  24,00  141,00  493,00  529,62  1.648,13  1.237,27  186,39
4          1114        ACEH JAYA  479,00  1.042,00    452,00   43,00   43,00  51,00  101,00  115,0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import re
import numpy as np, pandas as pd, geopandas as gpd
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/Logika Ui')
HASIL = ROOT / 'KTI_KARHUTLA' / 'hasil'
LAIN = ROOT / 'KTI_KARHUTLA' / 'lainnya'

def angka(x):
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    s = str(x).strip().replace('.', '').replace(',', '.')
    try:
        return float(s)
    except ValueError:
        return np.nan

def huruf(s):
    return re.sub(r'[^A-Z]', '', str(s).upper())

def pecah(s):
    s = str(s).strip().upper()
    kota = bool(re.match(r'^KOTA\b', s))
    s = re.sub(r'^(KOTA ADMINISTRASI|KOTA|KABUPATEN|KAB\.)\s+', '', s)
    return huruf(s), kota

# kode provinsi Kemendagri -> NAME_1 GADM (provinsi baru Papua dilebur ke Papua / Papua Barat)
PROV = {11: 'Aceh', 12: 'Sumatera Utara', 13: 'Sumatera Barat', 14: 'Riau', 15: 'Jambi',
        16: 'Sumatera Selatan', 17: 'Bengkulu', 18: 'Lampung', 19: 'Bangka Belitung',
        21: 'Kepulauan Riau', 31: 'Jakarta Raya', 32: 'Jawa Barat', 33: 'Jawa Tengah',
        34: 'Yogyakarta', 35: 'Jawa Timur', 36: 'Banten', 51: 'Bali',
        52: 'Nusa Tenggara Barat', 53: 'Nusa Tenggara Timur', 61: 'Kalimantan Barat',
        62: 'Kalimantan Tengah', 63: 'Kalimantan Selatan', 64: 'Kalimantan Timur',
        65: 'Kalimantan Utara', 71: 'Sulawesi Utara', 72: 'Sulawesi Tengah',
        73: 'Sulawesi Selatan', 74: 'Sulawesi Tenggara', 75: 'Gorontalo', 76: 'Sulawesi Barat',
        81: 'Maluku', 82: 'Maluku Utara', 91: 'Papua', 92: 'Papua Barat',
        93: 'Papua', 94: 'Papua', 95: 'Papua', 96: 'Papua Barat'}

def baca(nama, sheet='data'):
    d = pd.read_excel(LAIN / nama, sheet_name=sheet)
    d.columns = [str(c).strip() for c in d.columns]
    d = d.rename(columns={d.columns[0]: 'kode', d.columns[1]: 'nama_asli'})
    d['kode'] = pd.to_numeric(d['kode'], errors='coerce')
    d = d.dropna(subset=['kode']).copy()
    d['kode'] = d['kode'].astype(int)
    d = d.melt(id_vars=['kode', 'nama_asli'], var_name='tahun', value_name='luas_ha')
    d['tahun'] = pd.to_numeric(d['tahun'], errors='coerce')
    d = d.dropna(subset=['tahun']).copy()
    d['tahun'] = d['tahun'].astype(int)
    d['luas_ha'] = d['luas_ha'].map(angka)
    return d

# ---------- SiPongi kab/kota -> GID_2 GADM ----------
s = baca('luas-karhutla-kota.xlsx')
s['nama'] = s['nama_asli'].map(lambda x: pecah(x)[0])
s['kota'] = s['nama_asli'].map(lambda x: pecah(x)[1])
s['prov'] = (s['kode'] // 100).map(PROV).fillna('').map(huruf)

g = gpd.read_file(ROOT / 'GADM 2' / 'gadm41_IDN_2.shp')
g = pd.DataFrame(g.drop(columns='geometry'))
tipe = (g['ENGTYPE_2'].astype(str) if 'ENGTYPE_2' in g else pd.Series('', index=g.index)) + ' ' + \
       (g['TYPE_2'].astype(str) if 'TYPE_2' in g else pd.Series('', index=g.index))
g['kota'] = tipe.str.lower().str.contains('kota|city')
g['prov'] = g['NAME_1'].map(huruf)
g['nama'] = g['NAME_2'].map(lambda x: pecah(x)[0])

a = s[['kode', 'nama_asli', 'prov', 'nama', 'kota']].drop_duplicates('kode')
peta = a.merge(g[['GID_2', 'prov', 'nama', 'kota']], on=['prov', 'nama', 'kota'], how='left')
tidak = peta['GID_2'].isna()
unik = g.groupby(['prov', 'nama']).filter(lambda d: len(d) == 1)[['GID_2', 'prov', 'nama']]
cad = peta.loc[tidak, ['kode', 'nama_asli', 'prov', 'nama', 'kota']].merge(unik, on=['prov', 'nama'], how='left')
peta = pd.concat([peta[~tidak], cad], ignore_index=True)
dobel = peta['kode'].duplicated(keep=False)
peta = peta[~dobel]
print('SiPongi kab/kota:', len(a), '| cocok ke GADM:', int(peta['GID_2'].notna().sum()),
      '| ambigu dibuang:', int(dobel.sum()))
print(peta.loc[peta['GID_2'].isna(), ['kode', 'nama_asli']].head(30).to_string(index=False))

# ---------- FIRMS tahunan dari tabel gabungan ----------
f = pd.read_csv(HASIL / 'tabel_gabungan_nasional_semua_bulan.csv', parse_dates=['tanggal'],
                usecols=['tanggal', 'GID_2', 'NAME_1', 'titik_panas', 'frp_total'])
f['tahun'] = f['tanggal'].dt.year
fa = f.groupby(['GID_2', 'tahun'], as_index=False)[['titik_panas', 'frp_total']].sum()

d = (s[['kode', 'tahun', 'luas_ha']].merge(peta[['kode', 'GID_2']].dropna(), on='kode')
       .merge(fa, on=['GID_2', 'tahun']))
d.to_csv(HASIL / 'validasi_sipongi_firms_kabtahun.csv', index=False)

def rho(x, y):
    return round(x.corr(y, method='spearman'), 3)

d6 = d[d['tahun'].between(2020, 2025)]
print('\nKAB/KOTA-TAHUN 2020-2025: n =', len(d6), '| kab/kota =', d6['GID_2'].nunique())
print('Spearman luas karhutla vs titik panas:', rho(d6['luas_ha'], d6['titik_panas']))
print('Spearman luas karhutla vs FRP total  :', rho(d6['luas_ha'], d6['frp_total']))
baris = [(t, len(x), rho(x['luas_ha'], x['titik_panas']), rho(x['luas_ha'], x['frp_total']))
         for t, x in d.groupby('tahun')]
per = pd.DataFrame(baris, columns=['tahun', 'n', 'rho_titik', 'rho_frp'])
per['catatan'] = np.where(per['tahun'] == 2026, 'sebagian tahun', '')
print(per.to_string(index=False))

# ---------- tingkat provinsi ----------
p = baca('luas-karhutla-provinsi.xlsx')
p['NAME_1'] = p['kode'].map(PROV)
pp = p.dropna(subset=['NAME_1']).groupby(['NAME_1', 'tahun'], as_index=False)['luas_ha'].sum()
fp = f.groupby(['NAME_1', 'tahun'], as_index=False)[['titik_panas', 'frp_total']].sum()
dp = pp.merge(fp, on=['NAME_1', 'tahun'])
dp6 = dp[dp['tahun'].between(2020, 2025)]
print('\nPROVINSI-TAHUN 2020-2025: n =', len(dp6), '| provinsi =', dp6['NAME_1'].nunique())
print('Spearman luas karhutla vs titik panas:', rho(dp6['luas_ha'], dp6['titik_panas']))
print('Spearman luas karhutla vs FRP total  :', rho(dp6['luas_ha'], dp6['frp_total']))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SiPongi kab/kota: 514 | cocok ke GADM: 485 | ambigu dibuang: 0
 kode                        nama_asli
 7211                     BANGGAI LAUT
 7415                    BUTON SELATAN
 7414                     BUTON TENGAH
 7109 KEPULAUAN SIAU TAGULANDANG BIARO
 8103               KEPULAUAN TANIMBAR
 7411                     KOLAKA TIMUR
 7412                 KONAWE KEPULAUAN
 6302                         KOTABARU
 1277             KOTA PADANGSIDEMPUAN
 6411                      MAHAKAM ULU
 5321                           MALAKA
 7606                    MAMUJU TENGAH
 9211                MANOKWARI SELATAN
 6102                         MEMPAWAH
 7212                   MOROWALI UTARA
 7413                       MUNA BARAT
 1613                 MUSI RAWAS UTARA
 7504                         PAHUWATO
 1215                    PAKPAK BHARAT
 3218                      P

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np, pandas as pd
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/Logika Ui')
HASIL = ROOT / 'KTI_KARHUTLA' / 'hasil'
PRIORITAS = ['Riau', 'Jambi', 'Sumatera Selatan', 'Kalimantan Barat', 'Kalimantan Tengah', 'Kalimantan Selatan']

tabel = pd.read_csv(HASIL / 'tabel_gabungan_nasional_semua_bulan.csv', parse_dates=['tanggal'],
                    usecols=['tanggal', 'GID_2', 'NAME_1', 'titik_panas'])
z = np.load(HASIL / 'prob_final_nasional.npz')
assert len(z['PC']) == len(tabel), 'urutan baris tidak sama dengan tabel'
tabel['p_K1'] = z['PC'][:, 1]      # model C (sebelum jendela hujan)

gam = pd.read_csv(HASIL / 'persen_gambut_kabkota.csv').set_index('GID_2')['persen_gambut']
r_gam = gam.rank(pct=True)
tabel['persen_gambut'] = tabel['GID_2'].map(gam).fillna(0.0)
tabel['r_gambut'] = tabel['GID_2'].map(r_gam).fillna(0.0)

tabel = tabel.sort_values(['GID_2', 'tanggal']).reset_index(drop=True)
tabel['api_7h'] = (tabel.groupby('GID_2')['titik_panas']
                   .transform(lambda s: s.shift(1).rolling(7, min_periods=4).sum()))

tabel['r_awan'] = tabel.groupby('tanggal')['p_K1'].rank(pct=True)
tabel['r_api'] = tabel.groupby('tanggal')['api_7h'].rank(pct=True)
tabel['skor_tanpa_gambut'] = tabel['r_awan'] * tabel['r_api']
tabel['skor'] = tabel['skor_tanpa_gambut'] * tabel['r_gambut']
tabel['enam_prov'] = tabel['NAME_1'].isin(PRIORITAS)

simpan = ['tanggal', 'GID_2', 'NAME_1', 'p_K1', 'api_7h', 'persen_gambut', 'skor_tanpa_gambut', 'skor', 'enam_prov']
tabel[simpan].to_csv(HASIL / 'prioritas_harian.csv', index=False, float_format='%.4f')

hari = tabel.loc[tabel['api_7h'].notna(), 'tanggal'].max()
h = tabel[tabel['tanggal'] == hari].sort_values('skor', ascending=False)
cols = ['GID_2', 'NAME_1', 'p_K1', 'api_7h', 'persen_gambut', 'skor_tanpa_gambut', 'skor']
print('tanggal:', hari.date(), '\nNASIONAL')
print(h[cols].head(10).round(3).to_string(index=False))
print('\nENAM PROVINSI')
print(h[h['enam_prov']][cols].head(10).round(3).to_string(index=False))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
tanggal: 2026-10-14 
NASIONAL
      GID_2            NAME_1  p_K1  api_7h  persen_gambut  skor_tanpa_gambut  skor
IDN.22.10_1       Papua Barat 0.359   483.0         21.351              0.866 0.821
 IDN.22.7_1       Papua Barat 0.336   268.0         46.004              0.814 0.797
 IDN.22.1_1       Papua Barat 0.321  1023.0          6.036              0.832 0.711
 IDN.22.2_1       Papua Barat 0.282   842.0          6.591              0.738 0.638
 IDN.8.10_1             Jambi 0.233   515.0         39.149              0.653 0.635
 IDN.14.6_1 Kalimantan Tengah 0.220  2506.0         24.085              0.656 0.625
 IDN.23.4_1             Papua 0.373    57.0          3.138              0.762 0.623
  IDN.8.6_1             Jambi 0.225   343.0         44.898              0.633 0.619
IDN.12.11_1  Kalimantan Barat 0.383    23.0          6.569              0.684 0.590
 